In [3]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 441.8 kB/s eta 0:00:00m eta 0:00:010:00:02


In [9]:
import mysql.connector
import pandas as pd
import os

In [1]:


class mysql_db_helper:
    def __init__(self,data = 'ecomm'):
        database_name = data
        self.create_connection({"database_name":database_name})

    def create_connection(self, data):
        try:
            # refer to understand the connection https://www.red-gate.com/simple-talk/databases/mysql/retrieving-mysql-data-python/
            self.conn = mysql.connector.connect(option_files = '/home/de24/config/mysqldbconnectors.cnf')
            self.conn.database = data['database_name']
            #self.conn.autocommit = True
            self.curr = self.conn.cursor()
        except Exception as e:
            raise Exception(f"Error => {e}")
            

    def query_exec(self,query):
        try:
            self.curr.execute(query)          
            # 1. Check success
            print(f"Rows affected: {self.curr.rowcount}")
            # 2. Save the changes
            self.curr.commit() 
            print("Changes committed to the database.")
        except mysql.connector.Error as err:
            # 3. If an error occurs, the execution stops and goes here
            self.conn.rollback()
            # print(f"Error => {err}")
 raise Exception(f"Error => {err}")

    def query_exec_getresult(self,query):
        try:
            result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame
        except Exception as e:
            #raise Exception(f"Error => {e}")
            print(f"Error => {e}")
            return -1
        return result_df

    def connection_close(self):
        self.conn.close()

    def __del__(self):
        self.conn.close() 

In [4]:
import os

folder_path = f"/home/de24/S3_BUCKET/RAW/ecomm/customer/{20260303}/"
os.makedirs(folder_path, exist_ok=True)

mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'SELECT * FROM ecomm.customer LIMIT 10;')
df.head()
df.to_csv(f"{folder_path}/data.csv", index=False)
mysql_obj.connection_close()

/tmp/ipykernel_4486/850965339.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


In [28]:
mysql_obj = mysql_db_helper('ecomm')
mysql_obj.query_exec(f'''
CREATE TABLE ecomm.metadata_config (
    table_name VARCHAR(32),
    last_extracxt_date TIMESTAMP,
    duration INTEGER(2),
    duration_type VARCHAR(10)
)
''')
mysql_obj.connection_close()

Error => 1050 (42S01): Table 'metadata_config' already exists


In [4]:
mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SHOW TABLES;''')
mysql_obj.connection_close()
df.head(10)


/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,Tables_in_ecomm
0,customer
1,geolocation
2,metadata_config
3,order_items
4,order_payments
5,order_reviews
6,orders
7,roduct_category_name_translation
8,sellers


In [33]:
mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SELECT YEAR(order_purchase_timestamp),MONTH(order_purchase_timestamp),count(*) 
                                        FROM orders  
                                        GROUP BY YEAR(order_purchase_timestamp),MONTH(order_purchase_timestamp)
                                        ORDER BY YEAR(order_purchase_timestamp),MONTH(order_purchase_timestamp);''')
mysql_obj.connection_close()
df.head(40)

/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,YEAR(order_purchase_timestamp),MONTH(order_purchase_timestamp),count(*)
0,2016,9,4
1,2016,10,324
2,2016,12,1
3,2017,1,800
4,2017,2,1780
5,2017,3,2682
6,2017,4,2404
7,2017,5,3700
8,2017,6,3245
9,2017,7,4026


In [34]:
mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SELECT YEAR(shipping_limit_date),MONTH(shipping_limit_date),count(*)
                                        FROM order_items  
                                        GROUP BY YEAR(shipping_limit_date),MONTH(shipping_limit_date)
                                        ORDER BY YEAR(shipping_limit_date),MONTH(shipping_limit_date);''')
mysql_obj.connection_close()
df.head(10)

/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,YEAR(shipping_limit_date),MONTH(shipping_limit_date),count(*)
0,2016,9,4
1,2016,10,365
2,2016,12,1
3,2017,1,681
4,2017,2,1866
5,2017,3,2751
6,2017,4,2364
7,2017,5,4150
8,2017,6,3801
9,2017,7,4116


## So we shall plan to load the Data month wise 

In [8]:
mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SELECT * 
                                        FROM orders  
                                        WHERE YEAR(order_purchase_timestamp) = 2016 
                                        AND MONTH(order_purchase_timestamp) = 9;''')
mysql_obj.connection_close()
df.head(10)

/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaT,NaT,2016-10-28
1,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20
2,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,NaT,NaT,2016-09-30
3,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [26]:
# logic to load the Customer data in chunk in s3 bucket

year = '2017'
month = '07'

folder_path = f"/home/de24/S3_BUCKET/RAW/ecomm/customer/{year}{month}/"
os.makedirs(folder_path, exist_ok=True)

mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SELECT * FROM ecomm.customer 
                                        WHERE customer_id in (
                                            SELECT customer_id 
                                            FROM orders  
                                            WHERE YEAR(order_purchase_timestamp) = {year} 
                                            AND MONTH(order_purchase_timestamp) = {int(month)}
                                        );''')

df.to_csv(f"{folder_path}/data.csv", index=False)
mysql_obj.connection_close()
df.head()

/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,aa9f03ecd3728c9bd12e6d962c66c7cb,b03e9d9818ee170e9d6b983803c7d406,75388,trindade,GO
1,caa060324a30054bae148e4d7ad1da22,05aa2ef3623cdba9ca3e83b91f1eee5c,12970,piracaia,SP
2,bd30c8ec7f7139af7195e1f7f031e131,237c219a24b6a3183a216a5ca00f6edf,89031,blumenau,SC
3,872a25f014b99c4c17341cd9ad0d468f,636f22218f8430a8463c5ddc5f68fc62,14802,araraquara,SP
4,ada5afd2b1b1258fd60f507e92a63472,fbd58f38cbd2989e434b148f479534c8,35774,paraopeba,MG


In [29]:
mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''SELECT * 
                                        FROM order_items  
                                        limit 10;''')
mysql_obj.connection_close()
df.head(10)

/tmp/ipykernel_2297/4006831185.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40
